## PHME 2022 Data Challenge - Clean Solution

This is the restructured solution using clean architecture and modular components.

The solution defines three functions for the three classification tasks:
1. `classification_1`: Defect detection from SPI data
2. `classification_2`: Operator label classification (Good/Bad)
3. `classification_3`: Repair label classification (FalseScrap/NotPossibleToRepair)

In [ ]:
import sys
sys.path.append('..')

from src.utils.config_loader import ConfigLoader
from src.inference.predictor import PHMEPredictor

In [ ]:
# Load configuration and initialize predictor
config = ConfigLoader('../config/config.yaml')
predictor = PHMEPredictor(config)

# Load all trained models
predictor.load_models('../models')

In [ ]:
def classification_1(spi):
    """
    Task 1: Defect Detection from SPI data.
    
    Input: SPI DataFrame
    Output: List of tuples (PanelID, FigureID, ComponentID) for detected defects
    """
    return predictor.classification_1(spi)

In [ ]:
def classification_2(spi, aoi):
    """
    Task 2: Operator Label Classification (Good/Bad).
    
    Input: 
        - spi: SPI DataFrame
        - aoi: AOI DataFrame (without OperatorLabel and RepairLabel)
    Output: List of tuples (PanelID, FigureID, ComponentID, PredictedOperatorLabel)
    """
    return predictor.classification_2(spi, aoi)

In [ ]:
def classification_3(spi, aoi):
    """
    Task 3: Repair Label Classification (FalseScrap/NotPossibleToRepair).
    
    Input:
        - spi: SPI DataFrame
        - aoi: AOI DataFrame (with OperatorLabel, without RepairLabel)
    Output: List of tuples (PanelID, FigureID, ComponentID, PredictedRepairLabel)
    """
    return predictor.classification_3(spi, aoi)

## Test the Solution

The following code tests the solution with the training data to verify it works correctly.

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import statistics
import glob

# Load SPI data
dfs = []
for f in glob.glob("../data/SPI_*.csv.zip"):
    dfs.append(pd.read_csv(f, low_memory=False))
SPI = pd.concat(dfs, ignore_index=True)

# Load AOI data
dfs = []
for f in glob.glob("../data/AOI_*.csv.zip"):
    dfs.append(pd.read_csv(f))
AOI = pd.concat(dfs, ignore_index=True)

print(f"SPI data: {SPI.shape}")
print(f"AOI data: {AOI.shape}")

In [ ]:
# Run predictions
print("Running Task 1...")
results_1 = set(classification_1(SPI.copy()))

print("Running Task 2...")
results_2 = classification_2(
    SPI, 
    AOI[["PanelID", "FigureID", "MachineID", "ComponentID", "PinNumber", "AOILabel"]]
)

print("Running Task 3...")
results_3 = classification_3(
    SPI,
    AOI[["PanelID", "FigureID", "MachineID", "ComponentID", "PinNumber", "AOILabel", "OperatorLabel"]]
)

print("\nPredictions completed!")

In [ ]:
# Evaluate Task 1
groundtruth_1 = {tuple([str(f) for f in e]) for e in AOI[["PanelID", "FigureID", "ComponentID"]].values}
precision_1 = len(results_1 & groundtruth_1) / len(results_1) if len(results_1) > 0 else 0
recall_1 = len(results_1 & groundtruth_1) / len(groundtruth_1) if len(groundtruth_1) > 0 else 0
f1_1 = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if precision_1 + recall_1 > 0 else 0

# Evaluate Task 2
results_dict_2 = {(str(p), str(f), str(c)): l for p, f, c, l in results_2}
validationdata_2 = []
for t in AOI.drop_duplicates(subset=["PanelID", "FigureID", "ComponentID"], keep="first").itertuples():
    predicted = results_dict_2.get((str(t.PanelID), str(t.FigureID), str(t.ComponentID)), "-")
    validationdata_2.append((t.PanelID, t.FigureID, t.ComponentID, t.OperatorLabel, predicted))
validationdata_2 = pd.DataFrame(validationdata_2, columns=["PanelID", "FigureID", "ComponentID", "Real", "Predicted"])
f1_2 = classification_report(validationdata_2["Real"], validationdata_2["Predicted"], output_dict=True)["Bad"]["f1-score"]

# Evaluate Task 3
results_dict_3 = {(str(p), str(f), str(c)): l for p, f, c, l in results_3}
validationdata_3 = []
for t in AOI[AOI["RepairLabel"].isin({"FalseScrap", "NotPossibleToRepair"})].drop_duplicates(
    subset=["PanelID", "FigureID", "ComponentID"], keep="first"
).itertuples():
    predicted = results_dict_3.get((str(t.PanelID), str(t.FigureID), str(t.ComponentID)), "-")
    validationdata_3.append((t.PanelID, t.FigureID, t.ComponentID, t.RepairLabel, predicted))
validationdata_3 = pd.DataFrame(validationdata_3, columns=["PanelID", "FigureID", "ComponentID", "Real", "Predicted"])
cr = classification_report(validationdata_3["Real"], validationdata_3["Predicted"], output_dict=True)
f1_3 = (cr["FalseScrap"]["f1-score"] + cr["NotPossibleToRepair"]["f1-score"]) / 2

print("=" * 80)
print("RESULTS")
print("=" * 80)
print(f"F1 Score Task 1: {f1_1:.4f}")
print(f"F1 Score Task 2: {f1_2:.4f}")
print(f"F1 Score Task 3: {f1_3:.4f}")
print(f"Final Score: {statistics.mean([f1_1, f1_2, f1_3]):.4f}")
print("=" * 80)